In [1]:
import sys
sys.path.append('../')
import numpy as np
import scqubits.settings as settings
from joblib import Parallel, delayed
import utils_2Q_gate_zp as ut
import qutip as qt
settings.OVERLAP_THRESHOLD = 0.3

import pandas as pd
import scipy.sparse as ssp

## cz noise model

In [11]:
n_truc = 50
filter_ratio = 3

cz_run = True # True  # whether to use CZ gate or CNOT gate
import_2000_states = False # whether to import 2000 or 1000 states Hamiltonian
truc_one_qubit = 300 # truncation for single zero pi
n_full = 1000 # don't change this value, If import_2000 = True, n_full=2000, else 1000
# n_truc = 100 # <=n_full, Number of states to select from the full Hamiltonian

# 'short_path', 'all_path', 'hand_pick', 'cz_short_500_500_detune0' 
use_truc_model, truc_model_name = True, 'cz_short_500_500_detune1' 

# whether to calculate noisy fidelity
calculate_ideal, calculate_noise = True, True # False, True 
t1_tphi_other = 170 # μs
tg_list = [0] # [2, 9, 16, 23, 30] # Select the first row for testing

max_step_ideal = 1e-3 # Set max_step to 0 for parallel execution
nsteps_ideal = 1 / max_step_ideal  # Set nsteps to a large number for parallel execution
max_step_noisy = 1e-3 # Set max_step to 0 for parallel execution
nsteps_noisy = 1/ max_step_noisy  # Set nsteps to a large number for parallel execution
option_ideal =qt.Options(max_step=max_step_ideal, 
                            nsteps=nsteps_ideal, num_cpus=1)  
option_noisy =qt.Options(max_step=max_step_noisy, 
                            nsteps=nsteps_noisy, num_cpus=1)  

num_cpus, n_job = 16, len(tg_list) # Number of CPUs and jobs for parallel processing
[hspace_full, eket_tot, eval_tot, n_theta0_dress, 
    n_theta1_dress, hspace_0, hspace_1, logi_state
    ] = ut.load_qubit_data_2q(n_full, import_2000_states, truc_one_qubit)
dim_0 = len(hspace_0)
dim_1 = len(hspace_1) 
params = ut.load_drive_params_2q(cz_run)[tg_list, ]  # [1::4,] # Load pulse parameters from CSV

if cz_run: # CZ
    drive_term = n_theta1_dress
    W_20_50 = ( eval_tot[hspace_full.index('5-0')] - 
                eval_tot[hspace_full.index('2-0')] )
    print(f'Using CZ gate with W_20_50 = {W_20_50}')

if use_truc_model:
    if cz_run:
        hspace_select = ut.cz_truc_model[truc_model_name][:n_truc]
    # else:
    #     hspace_select = ut.cnot_truc_model[truc_model_name][:n_truc]
else:
    hspace_select = hspace_full[:n_truc]
index_select = [hspace_full.index(i) for i in hspace_select]
H_drive_select, eket_tot = ut.build_hamiltonian_2q(cz_run, index_select, eval_tot, 
                                                    eket_tot, drive_term)
logi_idx_select = [hspace_select.index(i) for i in logi_state]


Using CZ gate with W_20_50 = 15.982875383444132


In [14]:
len(hspace_0), len(hspace_1)

(154, 155)

In [12]:
[n_theta0, n_theta1, gamma_dephase_02_q0, gamma_dephase_02_q1
] = ut.load_noise_data_2q(t1_tphi_other) 

Gamma = 1 / 1e3 / t1_tphi_other
Gamma_decay_q0 = Gamma / (n_theta0[4,8]**2)
Gamma_decay_q1 = Gamma / (n_theta1[4,8]**2)
transition_a, n_theta0_trunc = ut.get_transitions_for_collapse(hspace_0, n_theta0, 
                                                            filter_ratio=filter_ratio)
transition_b, n_theta1_trunc = ut.get_transitions_for_collapse(hspace_1, n_theta1, 
                                                            filter_ratio=filter_ratio)

In [13]:
# Construct collapse operators
c_op_list = ut.construct_c_ops_2q(dim_0, dim_1, n_theta0_trunc, n_theta1_trunc, 
                               gamma_dephase_02_q0, gamma_dephase_02_q1, eket_tot, 
                               Gamma_decay_q0, Gamma_decay_q1, 
                               transition_a, transition_b)     
print(f'filter_ratio = {filter_ratio}, np.shape(c_op_list) = {np.shape(c_op_list)}')     

filter_ratio = 3, np.shape(c_op_list) = (307, 50, 50)


In [10]:
=

SyntaxError: invalid syntax (1763773627.py, line 1)

## cz generate data

In [ ]:
# truc1, truc_tot, charge_pick = 150, 2000, True
truc1, truc_tot, charge_pick = 11, 15, True

if charge_pick:
    folder = f'../../data/3ncut_two_zeropi/truc1=150_truc2=1000_pick=True/'
    hspace_0 = pd.read_csv(folder+ 'hspace_0.txt').to_numpy().flatten()
    hspace_1 = pd.read_csv(folder+ 'hspace_1.txt').to_numpy().flatten()
else:
    hspace_0 = np.arange(truc1)
    hspace_1 = np.arange(truc1)

n0 = len(hspace_0)
n1 = len(hspace_1)

folder = f'../../data/3ncut_two_zeropi/truc1=500/'
n_theta0 = np.load(folder+'n_theta0.npy')
n_theta1 = np.load(folder+'n_theta1.npy')

n_theta0 = ut.truncate_2(n_theta0, hspace_0)
n_theta1 = ut.truncate_2(n_theta1, hspace_1)

folder_save = f'data/3ncut_two_zeropi/truc1={truc1}_truc2={truc_tot}_pick={charge_pick}/'
eval_tot = pd.read_csv(folder_save+ 'eval_tot.txt').to_numpy().flatten()
eket_tot = ssp.csr_matrix(np.load(folder_save+ 'eket_tot.npy'))

###  Get wavefunction overlap for the truncated dressed states
bare_state = [[qt.tensor(qt.basis(n0, i), qt.basis(n1, j))
                        for j in range(n1)]
                            for i in range(n0)]
arg = [bare_state, n0, n1]

ut.print_time()
print("ut.find_overlap..... Time:")


Current China Time: 2025-07-26 17:49:27.937191+08:00
ut.find_overlap..... Time:


In [ ]:
n0, n1

(79, 79)

In [ ]:
result = Parallel(n_jobs=10)(delayed(ut.find_overlap)(i, *arg) for i in eket_tot)
top_index = [result[i][0] for i in range(eval_tot.shape[0])]
top_overlap = [result[i][1] for i in range(eval_tot.shape[0])]
# np.save(folder_save+f'top_index.npy', top_index)
# np.save(folder_save+f'top_overlap.npy', top_overlap)
ut.print_time()
print("pd.DataFrame(top_overlap)..... Time:")

ValueError: dimension mismatch

In [ ]:

##############################################################################################
### Get the dressed states index
index_array = [] # array index in each qubit (# in hspace_0, hspace_1)
for i, index in enumerate(top_index):
    j=0
    while j < len(index):
        if index[j] not in index_array:
            index_array.append(index[j])
            break
        else:
            j+=1
        if j==10:
            index_array.append((0,0))
            print(i, 'need to further compare overlap')
hspace_full = [(str(idx[0])+'-'+str(idx[1])) for idx in index_array] # actual state index in each qubit


In [ ]:

n_theta0_dress = ssp.kron(n_theta0, ssp.identity(n1))
n_theta1_dress = ssp.kron(ssp.identity(n0), n_theta1)
n_theta0_dress = (eket_tot @ n_theta0_dress @ eket_tot.conj().T).todense()
n_theta1_dress = (eket_tot @ n_theta1_dress @ eket_tot.conj().T).todense()
print("folder_save.... Time:")
ut.print_time()

pd.DataFrame(hspace_full).to_csv(folder_save+ f'hspace_full.txt', sep=',', index=False, header=True)
np.save(folder_save+f'n_theta0_dress.npy', n_theta0_dress)
np.save(folder_save+f'n_theta1_dress.npy', n_theta1_dress)
print(f'len(hspace_full) = {len(hspace_full)}')
print(f'np.shape(n_theta0_dress) = {np.shape(n_theta0_dress)}')
print(f'np.shape(n_theta1_dress) = {np.shape(n_theta1_dress)}')

In [ ]:
truc1, truc_tot, charge_pick = 150, 2000, True
folder_save = f'data/3ncut_two_zeropi/truc1={truc1}_truc2={truc_tot}_pick={charge_pick}/'
top_index = np.load(folder_save+f'top_index.npy')
top_overlap = np.load(folder_save+f'top_overlap.npy')

In [ ]:
print(np.shape(top_index), np.shape(top_overlap))
for i in range(10):
    print(f'Index {i} : {top_index[i].tolist()}')
    print(f'Overlap {i} : {top_overlap[i].tolist()}')

(2000, 10, 2) (2000, 10)
Index 0 : [[0, 0], [1, 1], [1, 5], [5, 1], [0, 3], [3, 0], [1, 15], [15, 1], [3, 3], [1, 18]]
Overlap 0 : [0.9999396276111384, 0.010978840316384604, 0.00020859342328918154, 0.00018595517935744935, 0.00017913003487539042, 0.00016843403549357931, 0.00014158235817762478, 0.0001273569475394904, 0.0001214540316215545, 6.131406559714229e-05]
Index 1 : [[0, 1], [1, 0], [1, 3], [3, 1], [5, 0], [0, 5], [1, 7], [1, 6], [1, 11], [5, 3]]
Overlap 1 : [0.9505023002292486, 0.31032427195171314, 0.014771294646896957, 0.004970186207100581, 0.0004387465859966915, 0.0004386049639906444, 0.00042285152622057885, 0.0004063211031873048, 0.00034164781683976953, 0.0003107372972729744]
Index 2 : [[1, 0], [0, 1], [3, 1], [1, 3], [7, 1], [0, 5], [13, 1], [6, 1], [12, 1], [3, 5]]
Overlap 2 : [0.9505036069178411, 0.31032979987969544, 0.014696808560418644, 0.004636469460813697, 0.00040809486023080284, 0.00033380110190113206, 0.0003098518330600406, 0.0002605070276110101, 0.00021912118954814215

In [ ]:
arg_select = [H_drive_select, W_20_50, num_cpus, c_op_list, logi_idx_select, 
            option_ideal, option_noisy]

f_noise = Parallel(n_jobs=n_job)(delayed(ut.cz_fidelity_log_noise)
                                    (args_indep, *arg_select)
                                for args_indep in params)

ut.print_data(f'f_{t1_tphi_other}us_{n_truc}', f_noise)
ut.print_time()